## Setup

In [ ]:
%config InlineBackend.figure_format = "retina"
from cycler import cycler
from matplotlib import colormaps, rcParams
from pypalettes import load_cmap

_cmap = load_cmap("default_jama")
if _cmap.name not in colormaps:
    colormaps.register(_cmap, name=_cmap.name)

rcParams["savefig.dpi"] = 100
rcParams["figure.dpi"] = 100
rcParams["font.size"] = 14

# Use LaTeX for all text rendering
rcParams["text.usetex"] = True
rcParams["font.family"] = "serif"
rcParams["font.serif"] = ["Computer Modern"]

# Ticks on all sides, pointing inward
rcParams["xtick.direction"] = "in"
rcParams["ytick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["ytick.right"] = True

# Minor ticks
rcParams["xtick.minor.visible"] = True
rcParams["ytick.minor.visible"] = True

# Slightly thicker axes and lines for retina displays
rcParams["axes.linewidth"] = 1.2
rcParams["lines.linewidth"] = 1.5

# Legend
rcParams["legend.framealpha"] = 0.9
rcParams["legend.fontsize"] = 12

rcParams["image.cmap"] = _cmap.name
rcParams["axes.prop_cycle"] = cycler(color=_cmap.colors)

In [ ]:
import sys

REPO_ROOT = "/home/ssage/Karmma/Galaxy_KaRMMa"
sys.path.append(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np

from karmma.utils import imm_blocks, load_run

## User Inputs

Edit the values below, then run the rest of the notebook top to bottom.

In [ ]:
# ============================================================
# USER INPUTS — loaded from an untracked local file, so switching which
# runs you're comparing never touches tracked notebook source.
#
# First time: copy notebooks/metadata_runs.example.yaml to
# notebooks/metadata_runs.yaml (gitignored) and edit that copy.
# ============================================================
import os

import yaml

_runs_config_path = os.path.join(REPO_ROOT, "notebooks", "metadata_runs.yaml")
if not os.path.exists(_runs_config_path):
    raise FileNotFoundError(
        f"{_runs_config_path} not found. Copy "
        "notebooks/metadata_runs.example.yaml to notebooks/metadata_runs.yaml "
        "(gitignored) and edit it with your own runs."
    )
with open(_runs_config_path) as _f:
    _runs_config = yaml.safe_load(_f)

output_dirs = _runs_config["output_dirs"]
mock_dg_paths = _runs_config["mock_dg_paths"]
run_labels = _runs_config.get("run_labels")

assert len(output_dirs) == len(mock_dg_paths), (
    f"output_dirs ({len(output_dirs)}) and mock_dg_paths ({len(mock_dg_paths)}) "
    "must have the same length"
)
n_runs = len(output_dirs)

if run_labels is None:
    run_labels = [f"Run {i}" for i in range(n_runs)]
assert len(run_labels) == n_runs, "run_labels must match the number of runs"

# One consistent color per run, reused across every plot below
_palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
run_colors = [_palette[i % len(_palette)] for i in range(n_runs)]

# Width for the "Run" label column in printed tables, so labels of any length line up
label_w = max(len(label) for label in run_labels) + 2

## Load Runs

In [ ]:
runs = [
    load_run(d, mock, label, color)
    for d, mock, label, color in zip(
        output_dirs, mock_dg_paths, run_labels, run_colors, strict=False
    )
]

nuts_runs = [r for r in runs if r["type"] == "nuts"]
mclmc_runs = [r for r in runs if r["type"] == "mclmc"]
theta_runs = [r for r in runs if r["theta_samples"] is not None]

print(f"Loaded {len(runs)} runs: {len(nuts_runs)} NUTS, {len(mclmc_runs)} MCLMC")

## MCMC Diagnostics

### Summary Table

In [ ]:
col_w = 12
header = (
    f"{'Run':<{label_w}}{'Type':>{col_w}}{'Seed':>{col_w}}"
    f"{'Step size':>{col_w}}{'N samples':>{col_w}}"
)
print(header)
print("-" * len(header))
for r in runs:
    print(
        f"{r['label']:<{label_w}}{r['type']:>{col_w}}{r['seed']:>{col_w}}"
        f"{r['step_size']:>{col_w}.4f}{r['n_samples']:>{col_w}}"
    )

### NUTS Diagnostics

In [ ]:
if nuts_runs:
    col_w = 12
    header = (
        f"{'Run':<{label_w}}{'Divs(samp)':>{col_w}}{'Divs(warm)':>{col_w}}"
        f"{'Acc(samp)':>{col_w}}{'Acc(warm)':>{col_w}}"
    )
    print(header)
    print("-" * len(header))
    for r in nuts_runs:
        e = r["extra"]
        print(
            f"{r['label']:<{label_w}}{int(e['is_divergent'].sum()):>{col_w}}"
            f"{int(e['warmup_is_divergent'].sum()):>{col_w}}"
            f"{e['acceptance_rate'].mean():>{col_w}.4f}"
            f"{e['warmup_acceptance_rate'].mean():>{col_w}.4f}"
        )
else:
    print("No NUTS runs in this comparison — skipping NUTS diagnostics.")

#### Acceptance Rate

In [ ]:
if nuts_runs:
    fig, ax = plt.subplots(figsize=(10, 4))
    for r in nuts_runs:
        acc = r["extra"]["acceptance_rate"]
        ax.plot(acc, color=r["color"], linewidth=0.8, alpha=0.8, label=r["label"])
        ax.axhline(acc.mean(), color=r["color"], linestyle="--", linewidth=1.0, alpha=0.6)
    ax.set_xlabel("Sample")
    ax.set_ylabel("Acceptance rate")
    ax.set_xlim(0, max(r["n_samples"] for r in nuts_runs))
    ax.set_ylim(0, 1)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.tight_layout()
    plt.show()

#### Integration Steps

In [ ]:
if nuts_runs:
    all_unique = np.unique(
        np.concatenate([r["extra"]["num_integration_steps"] for r in nuts_runs])
    )
    x = np.arange(len(all_unique))
    bar_width = 0.8 / len(nuts_runs)

    fig, ax = plt.subplots(figsize=(8, 4))
    for i, r in enumerate(nuts_runs):
        counts = np.array(
            [(r["extra"]["num_integration_steps"] == s).sum() for s in all_unique]
        )
        offset = (i - (len(nuts_runs) - 1) / 2) * bar_width
        ax.bar(
            x + offset,
            counts,
            bar_width,
            label=r["label"],
            color=r["color"],
            edgecolor="k",
            linewidth=0.5,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(all_unique)
    ax.set_xlabel("Number of integration steps")
    ax.set_ylabel("Counts")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.tight_layout()
    plt.show()

### MCLMC Diagnostics

In [ ]:
if mclmc_runs:
    col_w = 14
    header = (
        f"{'Run':<{label_w}}{'L':>{col_w}}{'Step size':>{col_w}}"
        f"{'L/step_size':>{col_w}}{'Nonans frac':>{col_w}}"
    )
    print(header)
    print("-" * len(header))
    for r in mclmc_runs:
        e = r["extra"]
        L = float(e["L"])
        step_size = float(r["step_size"])
        print(
            f"{r['label']:<{label_w}}{L:>{col_w}.4f}{step_size:>{col_w}.5f}"
            f"{L / step_size:>{col_w}.2f}{float(e['nonans'].mean()):>{col_w}.4f}"
        )
else:
    print("No MCLMC runs in this comparison — skipping MCLMC diagnostics.")

#### Energy Change

In [ ]:
if mclmc_runs:
    fig, ax = plt.subplots(figsize=(8, 4))
    for r in mclmc_runs:
        ax.plot(
            r["extra"]["energy_change"],
            color=r["color"],
            linewidth=0.7,
            alpha=0.7,
            label=r["label"],
        )
    ax.set_xlabel("Sample")
    ax.set_ylabel("Energy change (RMS)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Posterior Samples & Effective Sample Size

In [ ]:
for r in runs:
    er, ei = r["ess_xlm_real"], r["ess_xlm_imag"]
    print(
        f"{r['label']}:  xlm_real — min={er.min():.1f}, median={np.median(er):.1f}, mean={er.mean():.1f}"
        f"  |  xlm_imag — min={ei.min():.1f}, median={np.median(ei):.1f}, mean={ei.mean():.1f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for r in runs:
    axes[0].hist(
        r["ess_xlm_real"].flatten(),
        bins=30,
        color=r["color"],
        alpha=0.5,
        edgecolor="none",
        label=r["label"],
    )
    axes[0].axvline(r["ess_xlm_real"].min(), color=r["color"], linestyle="--", linewidth=1.0)
    axes[1].hist(
        r["ess_xlm_imag"].flatten(),
        bins=30,
        color=r["color"],
        alpha=0.5,
        edgecolor="none",
        label=r["label"],
    )
    axes[1].axvline(r["ess_xlm_imag"].min(), color=r["color"], linestyle="--", linewidth=1.0)

axes[0].set_xlabel("ESS")
axes[0].set_ylabel("Counts")
axes[0].set_title(r"$x_{\ell m}$ real")
axes[0].legend()
axes[1].set_xlabel("ESS")
axes[1].set_ylabel("Counts")
axes[1].set_title(r"$x_{\ell m}$ imag")
axes[1].legend()
plt.tight_layout()
plt.show()

### Theta ESS

In [ ]:
theta_param_names_latex = [r"$A_t$", r"$\log T$", r"$c$", r"$\log R$", r"$\mu_0$", r"$a$"]

if theta_runs:
    for r in theta_runs:
        ess_t = r["ess_theta"]
        print(
            f"{r['label']}: theta — min={ess_t.min():.1f}, median={np.median(ess_t):.1f}, "
            f"mean={ess_t.mean():.1f}"
        )

    nbins = theta_runs[0]["nbins"]
    x = np.arange(len(theta_param_names_latex))
    fig, axes = plt.subplots(1, nbins, figsize=(7 * nbins, 4), sharey=True)
    if nbins == 1:
        axes = [axes]

    distinct_n_samples = sorted({r["n_samples"] for r in theta_runs})

    for b, ax in enumerate(axes):
        for r in theta_runs:
            ax.scatter(x, r["ess_theta"][b], color=r["color"], label=r["label"], zorder=3, s=50)
        for n in distinct_n_samples:
            ax.axhline(
                n, color="k", linestyle="--", linewidth=1.0, label=f"$n_{{\\rm samples}}={n}$"
            )
        ax.set_xticks(x)
        ax.set_xticklabels(theta_param_names_latex)
        ax.set_xlabel(r"$\theta$ parameter")
        ax.set_ylabel("ESS")
        ax.set_title(f"Bin {b + 1}")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, bbox_to_anchor=(1.02, 0.5), loc="center left", borderaxespad=0
    )
    plt.suptitle(r"ESS per $\theta$ parameter")
    plt.tight_layout()
    plt.show()
else:
    print("No runs with theta samples — skipping theta ESS.")

### Theta Traces

In [ ]:
if theta_runs:
    theta_param_names_plain = ["A_t", "log_T", "c", "log_R", "mu0", "a"]
    n_params = len(theta_param_names_plain)
    nbins = theta_runs[0]["nbins"]
    max_n_samples = max(r["n_samples"] for r in theta_runs)

    truths_agree = all(
        np.allclose(r["true_theta"], theta_runs[0]["true_theta"]) for r in theta_runs
    )
    if not truths_agree:
        print(
            "Runs have different true_theta values (different mocks) — "
            "drawing one dotted truth line per run instead of a single shared one."
        )

    fig, axes = plt.subplots(
        n_params, nbins, figsize=(7 * nbins, 2.2 * n_params), sharex=True, squeeze=False
    )

    for b in range(nbins):
        for row in range(n_params):
            ax = axes[row, b]
            for r in theta_runs:
                ax.plot(
                    r["theta_samples"][:, b, row],
                    color=r["color"],
                    linewidth=0.7,
                    alpha=0.6,
                    label=r["label"],
                )
            if truths_agree:
                truth = theta_runs[0]["true_theta"][b, row]
                ax.axhline(
                    truth, color="k", linestyle="--", linewidth=1.0, label=f"Truth: {truth:.3f}"
                )
            else:
                for r in theta_runs:
                    ax.axhline(
                        r["true_theta"][b, row],
                        color=r["color"],
                        linestyle=":",
                        linewidth=1.0,
                        alpha=0.6,
                    )
            ax.set_ylabel(theta_param_names_latex[row])
            ax.set_title(rf"Bin {b + 1}, {theta_param_names_latex[row]}", fontsize=12)

    for b in range(nbins):
        axes[-1, b].set_xlabel("Sample")
        axes[-1, b].set_xlim(0, max_n_samples)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, bbox_to_anchor=(1.02, 0.5), loc="center left", borderaxespad=0
    )
    plt.suptitle(r"$\theta$ traces", y=1.01)
    plt.tight_layout()
    plt.show()

## Mass Matrix Conditioning

In [ ]:
col_w = 12
for kind_label, block_idx in [("real", 0), ("imag", 1)]:
    header = (
        f"{'Run':<{label_w}}{'Min':>{col_w}}{'Max':>{col_w}}{'Mean':>{col_w}}"
        f"{'Std':>{col_w}}{'Cond':>{col_w}}{'Inf/0':>{col_w}}"
    )
    print(f"xlm ({kind_label}) inverse mass matrix")
    print(header)
    print("-" * len(header))
    for r in runs:
        block = imm_blocks(r)[block_idx]
        bad = np.any(block == 0) or np.any(~np.isfinite(block))
        print(
            f"{r['label']:<{label_w}}{block.min():>{col_w}.2e}{block.max():>{col_w}.2e}"
            f"{block.mean():>{col_w}.2e}{block.std():>{col_w}.2e}"
            f"{block.max() / block.min():>{col_w}.2e}{str(bad):>{col_w}}"
        )
    print()

# phi is a whitened linear combination of all bins/parameters together (see
# theta_reparam), so unlike the xlm blocks it has no per-bin structure to
# break down — shown flat, labeled phi1..phiN.
n_phi = len(imm_blocks(runs[0])[2])
header = (
    f"{'Run':<{label_w}}"
    + "".join(f"{f'phi{k + 1}':>{col_w}}" for k in range(n_phi))
    + f"{'Inf/0':>{col_w}}"
)
print("phi (whitened theta) inverse mass matrix")
print(header)
print("-" * len(header))
for r in runs:
    phi_block = imm_blocks(r)[2]
    bad = np.any(phi_block == 0) or np.any(~np.isfinite(phi_block))
    row = "".join(f"{v:>{col_w}.2e}" for v in phi_block)
    print(f"{r['label']:<{label_w}}{row}{str(bad):>{col_w}}")

## Theta Corner Plots

In [ ]:
if theta_runs:
    from getdist import MCSamples, plots

    theta_param_names_plain = ["A_t", "log_T", "c", "log_R", "mu0", "a"]
    theta_labels_getdist = ["A_t", "log(T)", "c", "log(R)", "mu_0", "a"]
    nbins = theta_runs[0]["nbins"]
    line_args = [{"color": r["color"], "lw": 1.5} for r in theta_runs]

    # theta0 (the whitening reference point) should coincide across runs
    # sharing the same mock init; theta_runs[0]'s is used as the marker.
    theta0 = theta_runs[0]["theta_reparam"]["theta0"]

    for b in range(nbins):
        mc_list = [
            MCSamples(
                samples=r["theta_samples"][:, b],
                names=theta_param_names_plain,
                labels=theta_labels_getdist,
                label=r["label"],
            )
            for r in theta_runs
        ]
        g = plots.get_subplot_plotter()
        g.triangle_plot(
            mc_list,
            filled=True,
            colors=[r["color"] for r in theta_runs],
            line_args=line_args,
            markers={theta_param_names_plain[j]: theta0[b, j] for j in range(6)},
        )
        g.fig.suptitle(rf"Bin {b + 1} $\theta$ posteriors", y=1.01)
        plt.show()

## Log-Probability Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for r in runs:
    ax.plot(r["log_prob"], color=r["color"], linewidth=0.7, alpha=0.8, label=r["label"])
    ax.axhline(r["log_prob"].mean(), color=r["color"], linestyle="--", linewidth=1.0, alpha=0.6)
ax.set_xlabel("Sample")
ax.set_ylabel(r"$\log P$")
ax.set_xlim(0, max(r["n_samples"] for r in runs))
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()